### Singular Value Decomposition

So far in this lesson, you have gained some exposure to Singular Value Decomposition.  In this notebook, you will get some hands on practice with this technique.

Let's get started by reading in our libraries and setting up the data we will be using throughout this notebook

`1.` Run the cell below to create the **user_movie_subset** dataframe.  This will be the dataframe you will be using for the first part of this notebook.

In [1]:
# NumPy realiza a álgebra linear; pandas organiza os dados tabulares.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import libs.svd_tests as t
%matplotlib inline

# Carrega o catálogo de filmes e o histórico de avaliações.
movies = pd.read_csv('data/movies_clean.csv')
reviews = pd.read_csv('data/reviews_clean.csv')

# Remove as colunas de índice criadas durante a exportação dos CSVs.
del movies['Unnamed: 0']
del reviews['Unnamed: 0']

# Mantém somente as colunas necessárias para construir matrizes usuário-filme.
user_items = reviews[['user_id', 'movie_id', 'rating']]

# Para a primeira parte, filtra os três filmes antes do pivot para economizar memória.
subset_movie_ids = [75314, 68646, 99685]
user_movie_subset = (
    user_items[user_items['movie_id'].isin(subset_movie_ids)]
    .pivot(index='user_id', columns='movie_id', values='rating')
    .reindex(columns=subset_movie_ids)
    .dropna(axis=0)
)

# Exibe a pequena matriz completa que será decomposta nas próximas etapas.
print(user_movie_subset)

movie_id  75314  68646  99685
user_id                      
2213        7.0   10.0    8.0
2223        6.0   10.0    7.0
2942        8.0    9.0    8.0
3298        8.0   10.0   10.0
3424        9.0    9.0    9.0
5205        8.0    9.0    9.0


`2.` Now that you have the **user_movie_subset** matrix, use this matrix to correctly match each key to the correct value in the dictionary below.  Use the cells below the dictionary as necessary.

In [2]:
# Associa cada pergunta à alternativa correta calculada a partir do subconjunto.
a = 6
b = 68646
c = 'The Godfather'
d = 'Goodfellas'
e = 3298
f = 30685
g = 3

sol_1_dict = {
    'the number of users in the user_movie_subset': a,
    'the number of movies in the user_movie_subset': g,
    'the user_id with the highest average ratings given': e,
    'the movie_id with the highest average ratings received': b,
    'the name of the movie that received the highest average rating': c,
}


# Valida as cinco respostas com os testes fornecidos pelo exercício.
t.test1(sol_1_dict)

That's right!  There are 6 users in the dataset, which is given by the number of rows. There are 3 movies in the dataset given by the number of columns.  You can find the movies or users with the highest average ratings by taking the mean of each row or column.  Using the movies table, you can find the movie names associated with each id.


In [3]:
# A média por linha resume o padrão de notas de cada usuário.
user_average_ratings = user_movie_subset.mean(axis=1)
highest_rated_user = int(user_average_ratings.idxmax())

# A média por coluna resume as notas recebidas por cada filme.
movie_average_ratings = user_movie_subset.mean(axis=0)
highest_rated_movie = int(movie_average_ratings.idxmax())

# Remove movie_ids repetidos antes de consultar os três títulos.
subset_movie_names = (
    movies.drop_duplicates('movie_id').set_index('movie_id')
    .reindex(user_movie_subset.columns)['movie']
)

print('Usuário com maior média:', highest_rated_user)
print('Filme com maior média:', highest_rated_movie)
subset_movie_names

Usuário com maior média: 3298
Filme com maior média: 68646


movie_id
75314      Taxi Driver (1976)
68646    The Godfather (1972)
99685       Goodfellas (1990)
Name: movie, dtype: object

Now that you have a little more context about the matrix we will be performing Singular Value Decomposition on, we're going to do just that.  To get started, let's remind ourselves about the dimensions of each of the matrices we are going to get back.   Essentially, we are going to split the **user_movie_subset** matrix into three matrices:

$$ U \Sigma V^T $$


`3.` Given what you learned about in the previous parts of this lesson, provide the dimensions for each of the matrices specified above using the dictionary below.

In [4]:
# k é o número escolhido de fatores latentes; n e m são usuários e filmes.
a = 'a number that you can choose as the number of latent features to keep'
b = 'the number of users'
c = 'the number of movies'
d = 'the sum of the number of users and movies'
e = 'the product of the number of users and movies'

sol_2_dict = {
    'the number of rows in the U matrix': b,
    'the number of columns in the U matrix': a,
    'the number of rows in the V transpose matrix': a,
    'the number of columns in the V transpose matrix': c
}

# Confere se as dimensões seguem U(n x k) e V transposta(k x m).
t.test2(sol_2_dict)

That's right!  We will now put this to use, so you can see how the dot product of these matrices come together to create our user item matrix.  The number of latent features will control the sigma matrix as well, and this will a square matrix that will at most be the minimum of the number of users and number of movies (in our case the minimum is the 4 movies).


Now let's verify the above dimensions by performing SVD on our user-movie matrix.

`4.` Below you can find the code used to perform SVD in numpy.  You can see more about this functionality in the [documentation here](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.linalg.svd.html).  What do you notice about the shapes of your matrices?  If you try to take the dot product of the three objects you get back, can you directly do this to get back the user-movie matrix?

In [5]:
# full_matrices=True retorna U quadrada; os valores singulares vêm em um vetor.
u, s, vt = np.linalg.svd(user_movie_subset)
# A ordem mostra: sigma como vetor, U e V transposta.
s.shape, u.shape, vt.shape

((3,), (6, 6), (3, 3))

In [6]:
# Mostra a explicação fornecida pelo curso sobre o ajuste das dimensões.
t.question4thoughts()

Looking at the dimensions of the three returned objects, we can see the following:

 1. The u matrix is a square matrix with the number of rows and columns equaling the number of users. 

 2. The v transpose matrix is also a square matrix with the number of rows and columns equaling the number of items.

 3. The sigma matrix is actually returned as just an array with 3 values.  

 In order to set up the matrices in a way that they can be multiplied together, we have a few steps to perform: 

 1. Turn sigma into a square matrix with the number of latent features we would like to keep. 

 2. Change the columns of u and the rows of v transpose to match this number of dimensions. 

 If we would like to exactly re-create the user-movie matrix, we could choose to keep all of the latent features.


`5.` Use the thoughts from the above question to create **u**, **s**, and **vt** with three (the max number for this matrix) latent features.  When you have all three matrices created correctly, run the test below to show that the dot product of the three matrices creates the original user-movie matrix.  The matrices should have the following dimensions:

$$ U_{n x k} $$

$$\Sigma_{k x k} $$

$$V^T_{k x m} $$

where:

1. n is the number of users
2. k is the number of latent features to keep
3. m is the number of movies


In [7]:
# Mantém em U apenas as três colunas associadas aos valores singulares.
u_new = u[:, :3]

# np.linalg.svd retorna sigma como vetor; diag o transforma em matriz 3 x 3.
s_new = np.diag(s[:3])

# Como existem somente três filmes, todas as três linhas de V transposta são usadas.
vt_new = vt[:3, :]

In [8]:
# Verifica dimensões e confirma que U @ Sigma @ Vt reconstrói a matriz original.
assert u_new.shape == (6, 3), "Oops!  The shape of the u matrix doesn't look right. It should be 6 by 3."
assert s_new.shape == (3, 3), "Oops!  The shape of the sigma matrix doesn't look right.  It should be 3 x 3."
assert vt_new.shape == (3, 3), "Oops! The shape of the v transpose matrix doesn't look right.  It should be 3 x 3."
assert np.allclose(np.dot(np.dot(u_new, s_new), vt_new), user_movie_subset), "Oops!  Something went wrong with the dot product.  Your result didn't reproduce the original movie_user matrix."
print("That's right! The dimensions of u should be 6 x 3, and both v transpose and sigma should be 3 x 3.  The dot product of the three matrices how equals the original user-movie matrix!")

That's right! The dimensions of u should be 6 x 3, and both v transpose and sigma should be 3 x 3.  The dot product of the three matrices how equals the original user-movie matrix!


`6.` Scikit-learn also has an easy way to implement SVD.  The documentation for this implementation can be found [in the scikit-learn documentation here](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html).  Below, we have imported this library, as well as created the 1-0 user-movie matrix used earlier in this lesson. Use `fit_transform` on the `user_by_movie` matrix to obtain

$$ U \Sigma V^T $$

with 200 components and 5 iterations.

In [9]:
# TruncatedSVD opera diretamente sobre matrizes esparsas.
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

# Cria posições estáveis para os IDs nos eixos da matriz completa.
user_ids = np.sort(user_items['user_id'].unique())
movie_ids = np.sort(user_items['movie_id'].unique())
positive_interactions = user_items[user_items['rating'] > 0]
user_positions = pd.Categorical(
    positive_interactions['user_id'], categories=user_ids
).codes
movie_positions = pd.Categorical(
    positive_interactions['movie_id'], categories=movie_ids
).codes

# Cada avaliação positiva vira 1; ausências permanecem zeros implícitos na CSR.
user_by_movie = csr_matrix(
    (
        np.ones(len(positive_interactions), dtype=np.int8),
        (user_positions, movie_positions)
    ),
    shape=(len(user_ids), len(movie_ids))
)
user_by_movie.sum_duplicates()
user_by_movie.data[:] = 1

# Mantém 200 fatores latentes e fixa a semente para resultados reproduzíveis.
svd = TruncatedSVD(n_components=200, n_iter=5, random_state=42)

# fit_transform retorna U @ Sigma; components_ corresponde a V transposta.
u_sigma = svd.fit_transform(user_by_movie)
vt = svd.components_
s = np.diag(svd.singular_values_)

# Divide cada coluna por seu valor singular para recuperar U explicitamente.
u = u_sigma / svd.singular_values_
print('u', u.shape)
print('s', s.shape)
print('vt', vt.shape)

u (8022, 200)
s (200, 200)
vt (200, 13850)


`7.` How much variability can be explained by each of the 200 components?  How much of the variability can be explained in total by the 200 components?

In [10]:
# A razão explicada mede a parcela da variância capturada por cada fator.
explained_variance = pd.DataFrame({
    'component': np.arange(1, len(svd.explained_variance_ratio_) + 1),
    'explained_variance_ratio': svd.explained_variance_ratio_
})
explained_variance['cumulative_variance_ratio'] = (
    explained_variance['explained_variance_ratio'].cumsum()
)

# Exibe uma amostra por componente e o total coberto pelos 200 fatores.
display(explained_variance.head(10))
print(
    'Variância total explicada pelos 200 componentes: '
    f"{svd.explained_variance_ratio_.sum():.2%}"
)

,component,explained_variance_ratio,cumulative_variance_ratio
0,1,0.061238,0.061238
1,2,0.022930,0.084167
2,3,0.014273,0.098441
3,4,0.011024,0.109464
4,5,0.010385,0.119849
5,6,0.009735,0.129585
6,7,0.008604,0.138189
7,8,0.008120,0.146309
8,9,0.007426,0.153735
9,10,0.006598,0.160332


Variância total explicada pelos 200 componentes: 56.43%


`8.` Create your prediction matrix and verify it is the shape you would expect.

In [11]:
# Usa float32 para reduzir pela metade a memória da matriz densa de predições.
prediction_matrix = (
    u.astype(np.float32)
    @ s.astype(np.float32)
    @ vt.astype(np.float32)
)

# A reconstrução deve ter uma linha por usuário e uma coluna por filme.
expected_shape = user_by_movie.shape
assert prediction_matrix.shape == expected_shape
print('Formato esperado:', expected_shape)
print('Formato das predições:', prediction_matrix.shape)

Formato esperado: (8022, 13850)
Formato das predições: (8022, 13850)
